In [1]:
from __future__ import annotations

from pathlib import Path
import pandas as pd
import numpy as np

import dash
from dash import html, dcc, Input, Output, dash_table
import dash_bootstrap_components as dbc
import plotly.express as px


# =========================
# PATHS
# =========================
BASE_DIR = Path.cwd()
PROJECT_DIR = BASE_DIR if (BASE_DIR / "data").exists() else BASE_DIR.parent

PRO_DATA = PROJECT_DIR / "data" / "processed"
OUT_DIR = PROJECT_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

HOURLY_CSV = PRO_DATA / "air_quality_hourly_clean.csv"


# =========================
# LOAD DATA
# =========================
df = pd.read_csv(HOURLY_CSV, parse_dates=["datetime", "date"])
df["site"] = df["site"].astype(str)
df["date"] = pd.to_datetime(df["date"]).dt.normalize()

daily = (
    df.groupby(["site", "date"], as_index=False)
      .agg(
          pm10_mean=("pm10", "mean"),
          pm25_mean=("pm25", "mean"),
          pm10_obs=("pm10", "count"),
          pm25_obs=("pm25", "count"),
      )
)
daily[["pm10_mean", "pm25_mean"]] = daily[["pm10_mean", "pm25_mean"]].round(2)

SITES = sorted(df["site"].unique())
DATE_MIN = daily["date"].min()
DATE_MAX = daily["date"].max()


# =========================
# DASH APP
# =========================
external_stylesheets = [
    dbc.themes.BOOTSTRAP,
    "https://cdn.jsdelivr.net/npm/bootstrap-icons@1.11.3/font/bootstrap-icons.css",
]
app = dash.Dash(__name__, external_stylesheets=external_stylesheets)
app.title = "Bradford Winter Air Quality"



# =========================
# GLOBAL FONT
# =========================
FONT_FAMILY = '"Open Sans", "Inter", "Segoe UI", Arial, sans-serif'

HEADER_BG = "#276167"
HEADER_HEIGHT_PX = 150
SIDEBAR_WIDTH_PX = 300


STYLES = {
    "app": {"fontFamily": FONT_FAMILY, "backgroundColor": "white"},

    "header": {
        "position": "fixed",
        "top": 0,
        "left": 0,
        "right": 0,
        "height": f"{HEADER_HEIGHT_PX}px",
        "backgroundColor": HEADER_BG,
        "color": "white",
        "zIndex": 1100,
        "display": "flex",
        "alignItems": "center",
        "justifyContent": "center",
        "padding": "0 24px",
        "boxShadow": "0 2px 10px rgba(0,0,0,0.2)",
    },

    "logo": {"height": "85px", "marginLeft": "18"},

    "header_inner": {
        "width": "100%",
        "maxWidth": "1200px",
        "display": "flex",
        "alignItems": "center",
        "gap": "14px",
    },

    "title_block": {"display": "flex", "flexDirection": "column", "gap": "4px"},

    "title": {"fontSize": "34px", "fontWeight": 800, "margin": 0, "lineHeight": "1.0"},
    "subtitle": {"fontSize": "16px", "opacity": 0.95, "margin": 0},

    "sidebar": {
        "position": "fixed",
        "top": f"{HEADER_HEIGHT_PX}px",
        "left": 0,
        "bottom": 0,
        "width": f"{SIDEBAR_WIDTH_PX}px",
        "backgroundColor": "#f8f9fa",
        "padding": "16px",
        "borderRight": "1px solid #e5e5e5",
        "overflowY": "auto",
        "zIndex": 1050,
        "fontFamily": FONT_FAMILY,
    },

    "content": {
        "marginTop": f"{HEADER_HEIGHT_PX}px",
        "marginLeft": f"{SIDEBAR_WIDTH_PX}px",
        "padding": "20px",
        "backgroundColor": "white",
        "minHeight": "100vh",
        "fontFamily": FONT_FAMILY,
    },

    "card": {
        "borderRadius": "14px",
        "boxShadow": "0 2px 10px rgba(0,0,0,0.06)",
        "border": "1px solid #eee",
    },

    "label": {"fontWeight": 700, "marginTop": "12px", "marginBottom": "6px", "fontSize": "13px"},

    "filter_header": {
        "display": "flex",
        "alignItems": "center",
        "justifyContent": "space-between",
        "marginBottom": "8px",
    },
}


# =========================
# LAYOUT
# =========================
header = html.Div(
    style={
        **STYLES["header"],
        "backgroundColor": "#276167",
        "height": "140px",
        "position": "fixed",
        "top": "0",
        "left": "0",
        "right": "0",
        "zIndex": "1000",
    },
    children=[
        html.Div(
            style={
                **STYLES["header_inner"],
                "height": "120px",
                "display": "flex",
                "alignItems": "center",
                "justifyContent": "center",
                "position": "relative",
            },
            children=[
                # Logo (left)
                html.Img(
                    src=app.get_asset_url("bradford.png"),
                    style={
                        **STYLES["logo"],
                    },
                ), 
                # Title block (centered)
                html.Div(
                    style={
                        **STYLES["title_block"],
                        "textAlign": "center",
                        "color": "white",
                        "lineHeight": "1.2",
                    },
                    
                    children=[
                        html.H1(
                            "Winter Air Quality in Bradford",
                            style={
                                **STYLES["title"],
                                "fontSize": "42px",
                                "margin": "0",
                                "fontWeight": "700",
                            },
                        ),
                        html.P(
                            "Keighley · Tong Street · Treadwell Mills",
                            style={
                                **STYLES["subtitle"],
                                "fontSize": "18px",
                                "margin": "6px 0 0 0",
                                "color": "#E6F2F2",
                                "fontWeight": "500",
                            },
                        ),
                        html.P(
                            "Nov 2024 – Jan 2025",
                            style={
                                "fontSize": "15px",
                                "margin": "4px 0 0 0",
                                "color": "#D3E7E7",
                                "fontWeight": "400",
                            },
                        ),
                    ],
                ),
            ],
        )
    ],
)

sidebar = html.Div(
    style=STYLES["sidebar"],
    children=[
        html.Div(
            style=STYLES["filter_header"],
            children=[
                html.Div(
                    style={"display": "flex", "alignItems": "center", "gap": "8px"},
                    children=[
                        html.I(className="bi bi-funnel-fill"),
                        html.Span("Filters", style={"fontWeight": 800}),
                    ],
                ),
                dbc.Button(
                    [html.I(className="bi bi-arrow-counterclockwise"), html.Span(" Reset", style={"marginLeft": "6px"})],
                    id="btn-reset",
                    color="secondary",
                    outline=True,
                    size="sm",
                ),
            ],
        ),

        html.Div(style=STYLES["label"], children="Site"),
        dcc.Dropdown(
            id="site",
            options=[{"label": s, "value": s} for s in SITES],
            value=SITES,
            multi=True,
            clearable=False,
        ),

        html.Div(style=STYLES["label"], children="Date range"),
        dcc.DatePickerRange(
            id="date-range",
            start_date=DATE_MIN,
            end_date=DATE_MAX,
            min_date_allowed=DATE_MIN,
            max_date_allowed=DATE_MAX,
            display_format="DD/MM/YYYY",
            style={"width": "100%"},
        ),

        html.Div(style=STYLES["label"], children="Pollutant"),
        dcc.RadioItems(
            id="pollutant",
            options=[
                {"label": " PM2.5", "value": "PM2.5"},
                {"label": " PM10", "value": "PM10"},
            ],
            value="PM2.5",
            labelStyle={"display": "block", "marginBottom": "6px"},
        ),

        html.Hr(),
        
        dbc.Alert("Tip: Hover to compare sites on the same date.", color="info", style={"fontSize": "12px"}),
        dbc.Alert("Note: Daily means are computed from valid hourly observations.", color="warning", style={"fontSize": "12px"}),
    ],
)

content = html.Div(
    style=STYLES["content"],
    children=[
        dbc.Row(
            [
                dbc.Col(
                    dbc.Card(
                        dbc.CardBody(
                            [
                                html.H5("Overall daily trend"),
                                dcc.Graph(id="fig-overall", config={"displayModeBar": False}),
                            ]
                        ),
                        style=STYLES["card"],
                    ),
                    md=12,
                )
            ],
            className="g-3",
        ),

        dbc.Row(
            [
                dbc.Col(
                    dbc.Card(
                        dbc.CardBody(
                            [
                                html.H5("Daily trend by site"),
                                dcc.Graph(id="fig-by-site", config={"displayModeBar": False}),
                            ]
                        ),
                        style=STYLES["card"],
                    ),
                    md=12,
                )
            ],
            className="g-3",
            style={"marginTop": "10px"},
        ),

        dbc.Row(
            [
                dbc.Col(
                    dbc.Card(
                        dbc.CardBody(
                            [
                                html.H5("Site ranking"),
                                dcc.Graph(id="fig-ranking", config={"displayModeBar": False}),
                            ]
                        ),
                        style=STYLES["card"],
                    ),
                    md=12,
                )
            ],
            className="g-3",
            style={"marginTop": "10px"},
        ),

        dbc.Row(
            [
                dbc.Col(
                    dbc.Card(
                        dbc.CardBody(
                            [
                                html.H5("Daily summary table"),
                                dash_table.DataTable(
                                    id="tbl-daily",
                                    page_size=12,
                                    sort_action="native",
                                    filter_action="native",
                                    style_table={"overflowX": "auto"},
                                    style_header={"fontWeight": "800"},
                                    style_cell={
                                        "fontFamily": FONT_FAMILY,
                                        "fontSize": "12px",
                                        "padding": "6px",
                                        "whiteSpace": "nowrap",
                                    },
                                ),
                            ]
                        ),
                        style=STYLES["card"],
                    ),
                    md=12,
                )
            ],
            className="g-3",
            style={"marginTop": "10px"},
        ),
    ],
)

app.layout = html.Div(style=STYLES["app"], children=[header, sidebar, content])


# =========================
# HELPERS
# =========================
def filter_daily(selected_sites, start_date, end_date):
    d = daily.copy()
    if selected_sites:
        d = d[d["site"].isin(selected_sites)]
    if start_date is not None:
        d = d[d["date"] >= pd.to_datetime(start_date).normalize()]
    if end_date is not None:
        d = d[d["date"] <= pd.to_datetime(end_date).normalize()]
    return d


def polish(fig, title, y_title):
    fig.update_layout(
        title=title,
        hovermode="x unified",
        template="plotly_white",
        margin=dict(l=40, r=20, t=60, b=40),
        height=420,
        legend_title_text="",
        font=dict(family=FONT_FAMILY),
    )
    fig.update_xaxes(title_text="Date", tickfont=dict(size=13), title_font=dict(size=14))
    fig.update_yaxes(title_text=y_title, tickfont=dict(size=13), title_font=dict(size=14))
    return fig


# =========================
# CALLBACKS
# =========================
@app.callback(
    Output("site", "value"),
    Output("date-range", "start_date"),
    Output("date-range", "end_date"),
    Output("pollutant", "value"),
    Input("btn-reset", "n_clicks"),
    prevent_initial_call=True,
)
def reset_filters(_):
    return SITES, DATE_MIN, DATE_MAX, "PM2.5"


@app.callback(
    Output("fig-overall", "figure"),
    Output("fig-by-site", "figure"),
    Output("fig-ranking", "figure"),
    Output("tbl-daily", "data"),
    Output("tbl-daily", "columns"),
    Input("site", "value"),
    Input("date-range", "start_date"),
    Input("date-range", "end_date"),
    Input("pollutant", "value"),
)
def update(selected_sites, start_date, end_date, pollutant):
    d = filter_daily(selected_sites, start_date, end_date)

    if pollutant == "PM2.5":
        value_col = "pm25_mean"
        y_title = "PM2.5 (µg/m³)"
        overall_title = "Overall daily PM2.5"
        by_site_title = "Daily PM2.5 by site"
        rank_title = "Site ranking by average daily PM2.5"
    else:
        value_col = "pm10_mean"
        y_title = "PM10 (µg/m³)"
        overall_title = "Overall daily PM10 (mean across selected sites)"
        by_site_title = "Daily PM10 by site"
        rank_title = "Site ranking by average daily PM10"

    overall = d.groupby("date", as_index=False).agg(value=(value_col, "mean"))
    fig_overall = polish(px.line(overall, x="date", y="value"), overall_title, y_title)

    fig_by_site = polish(px.line(d, x="date", y=value_col, color="site"), by_site_title, y_title)

    ranking = (
        d.groupby("site", as_index=False)
         .agg(
             avg_daily=(value_col, "mean"),
             days_with_data=(value_col, lambda s: int(s.notna().sum())),
         )
    )
    ranking["avg_daily"] = ranking["avg_daily"].round(2)
    ranking = ranking.sort_values("avg_daily", ascending=False)

    fig_rank = px.bar(
        ranking,
        x="avg_daily",
        y="site",
        orientation="h",
        hover_data={"days_with_data": True},
        labels={"avg_daily": f"Average daily {pollutant} (µg/m³)", "site": "Site"},
        title=rank_title,
    )
    fig_rank.update_layout(template="plotly_white", height=420, margin=dict(l=40, r=20, t=60, b=40), font=dict(family=FONT_FAMILY))
    fig_rank.update_yaxes(categoryorder="total ascending", tickfont=dict(size=13), title_font=dict(size=14))
    fig_rank.update_xaxes(tickfont=dict(size=13), title_font=dict(size=14))

    table_cols = ["site", "date", "pm10_mean", "pm25_mean", "pm10_obs", "pm25_obs"]
    table = d[table_cols].copy()
    table["date"] = table["date"].dt.strftime("%d/%m/%Y")

    columns = [
        {"name": "Site", "id": "site"},
        {"name": "Date", "id": "date"},
        {"name": "PM10 daily mean", "id": "pm10_mean"},
        {"name": "PM2.5 daily mean", "id": "pm25_mean"},
        {"name": "PM10 observed (hours)", "id": "pm10_obs"},
        {"name": "PM2.5 observed (hours)", "id": "pm25_obs"},
    ]

    return fig_overall, fig_by_site, fig_rank, table.to_dict("records"), columns


# =========================
# RUN
# =========================
if __name__ == "__main__":
    app.run(debug=True, host="127.0.0.1", port=8051)